In [1]:
import geemap
import utils as F
import ee

In [2]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

import utils as F

In [3]:
roi = Map.user_roi
year = 2019
start_date, end_date = '2019-01-01', '2020-01-01'
MGRS_TILE = '30UYC'
cloud_limit = 40

def get_s2_sr_cld_col(aoi, start_date, end_date, CLOUDY_PIXEL_PERCENTAGE):
    # Import and filter S2 SR.
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
        .filterBounds(aoi)\
        .filterDate(start_date, end_date)\
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', CLOUDY_PIXEL_PERCENTAGE)))

    # Import and filter s2cloudless.
    s2_cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')\
        .filterBounds(aoi)\
        .filterDate(start_date, end_date))\
        

    # Join the filtered s2cloudless collection to the SR collection by the 'system:index' property.
    return ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': s2_sr_col,
        'secondary': s2_cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))
    
def add_cloud_bands(img):
    
     # Get s2cloudless image, subset the probability band.
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')
  
    #Condition s2cloudless by the probability threshold value.
    is_cloud = cld_prb.lte(60).rename('cloud')
  
    #Add the cloud probability layer and cloud mask as image bands.
    img =  img.addBands(ee.Image([cld_prb, is_cloud]))
      
    return img.updateMask(is_cloud).unmask(0)

def cal_cloud_per(img):
    
    cloud_img = img.select('cloud')
    Cloud_count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.sum(),
      geometry = roi,
      scale = 30,
      maxPixels = 1e13
    ).get('cloud')
  
    count = cloud_img.reduceRegion(
      
      reducer = ee.Reducer.count(),
      geometry = roi,
      scale = 30,
      maxPixels =  1e13
    ).get('cloud')
    
    cloud_per = ee.Number(Cloud_count).divide(count).multiply(100)
    
    return img.set('cloud_per', cloud_per)
    

def VI_cal_S2(img):
    NDWI = img.normalizedDifference(['B3','B8']).rename('NDWI')
    NDVI = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    MNDWI = img.normalizedDifference(['B3','B11']).rename('MNDWI')
    NDVI = img.normalizedDifference(['B8', 'B4']).rename('NDVI')                    
    TFI = img.expression('(100*B8)/(B4*(B8-B4+100))',
                          {
                          'B8': img.select('B8'),
                          'B4': img.select('B4')
                          }).rename('TFI')
    return img.addBands(NDWI).addBands(TFI).addBands(NDVI).addBands(MNDWI)

def TF_binary_cal(img):
    
    TFI = img.select('TFI').gt(0.1)#.and(img.select('TFI').lt(1))
    NDWI = img.select('NDWI').lt(0.2)
    TF_binary = TFI.And(NDWI)
    return TF_binary.rename('TF')

def CLIP(img):
    return img.clip(roi)

S2 = get_s2_sr_cld_col(roi, start_date, end_date, 30)
S2 = S2.filter(ee.Filter.eq('MGRS_TILE', '30UYD')).map(VI_cal_S2).map(add_cloud_bands)\
        .map(CLIP).map(cal_cloud_per)\
        .filter(ee.Filter.gt('cloud_per', cloud_limit))

# sum_TF = S2.map(TF_binary_cal).toBands().reduce(ee.Reducer.sum())
# sum_cloud = S2.select('cloud').toBands().reduce(ee.Reducer.sum())
# Frequency = sum_TF.divide(sum_cloud).rename('Freq')



In [4]:
def VI_cal_L8(img):
    
    NDVI = img.normalizedDifference(['SR_B5','SR_B4']).rename('NDVI')
    NDWI = img.expression('(green - nir)/(green + nir)',
                          {
                            'green': img.select('SR_B3'),
                            'nir'  : img.select('SR_B5')
                          }).rename('NDWI')

    TFI = img.expression('(100*B8)/(B4*(B8-B4+50))',
                          {
                          'B8': img.select('SR_B5'),
                          'B4': img.select('SR_B4')
                          }).rename('TFI')
                          
    return img.addBands(TFI).addBands(NDWI).addBands(NDVI)

def Cal_doy(img):
    doy = ee.Date(img.get('system:time_start')).getRelative('day', 'year').add(1)
    return img.set('DOY', doy)

def cloud_mask_L8(img):
    qa = img.select('QA_PIXEL')
    cloudShadowBitMask = 1 << 3
    cloudsBitMask = 1 << 5
    cirrusBitMask = 1 << 4
    dilatedCloudBitMask = 1 << 1
  
    cloud = qa.bitwiseAnd(cloudShadowBitMask).eq(0)\
                 .And(qa.bitwiseAnd(cloudsBitMask).eq(0))\
                 .And(qa.bitwiseAnd(cirrusBitMask).eq(0))\
                 .And(qa.bitwiseAnd(dilatedCloudBitMask).eq(0))
    
    return img.addBands(cloud.rename('cloud')).updateMask(cloud).unmask(0)

def Landsat_SR_corvert(image):
    opticalBands = image.select('SR_B.').multiply(0.275).add(-2000)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(opticalBands, None, True)\
                .addBands(thermalBands, None, True)

def Landsat_processing(Collection, data_start, data_end):
    Landsat_C = (Collection
                 .filterDate(data_start, data_end)
                 .filterBounds(roi)
                 .map(Landsat_SR_corvert)
                 .map(cloud_mask_L8)
                 .map(Cal_doy))

    doy_list = Landsat_C.aggregate_array('DOY').distinct()

    def make_unique_doy(doy):
        doy_num = ee.Number(doy)
        unique_doy = (Landsat_C
                      .filter(ee.Filter.eq('DOY', doy_num))
                      .mosaic()
                      .set('DOY', doy_num))
        return unique_doy

    # 在 Python API 中，map 的参数是一个 Python 函数
    unique_doy_collection = doy_list.map(make_unique_doy)

    return  ee.ImageCollection(unique_doy_collection).map(CLIP).map(cal_cloud_per).filter(ee.Filter.gt('cloud_per', 40))

def band_rename_L89(img):
    return img.select(['SR_B2','SR_B3','SR_B4','SR_B5','NDWI', 'TFI','NDVI', 'cloud'], ['B2','B3','B4','B8','NDWI', 'TFI','NDVI', 'cloud'])

L9_collect = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
L8_collect = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
L9 = Landsat_processing(L9_collect, start_date, end_date).map(VI_cal_L8)
L8 = Landsat_processing(L8_collect, start_date, end_date).map(VI_cal_L8) 

In [5]:
all_TF_collect = S2.merge(L8).merge(L9)
all_TF_collect = ee.ImageCollection(all_TF_collect)
sum_TF = all_TF_collect.map(TF_binary_cal).toBands().reduce(ee.Reducer.sum())
sum_cloud = all_TF_collect.select('cloud').toBands().reduce(ee.Reducer.sum())
Frequency = sum_TF.divide(sum_cloud).rename('Freq')
  

In [6]:
def Norm(img):
    tfi_mask = img.select('TFI').gt(0).And(img.select('TFI').lt(0.5))
    NDVI_mask = img.select('NDVI').gt(0)
    NDWI_mask = img.select('NDWI').lt(0)
    return img.mask(tfi_mask.And(NDVI_mask).And(NDWI_mask)).unmask(0)

def normalized_TFI(img):
    tfi_mask = img.select('TFI').gt(0).And(img.select('TFI').lt(0.5))
    ndvi_mask = img.select('NDVI').gt(0)
    ndwi_mask = img.select('NDWI').lt(0)
    zz = img.select('NDWI').mask(ndvi_mask.And(tfi_mask).And(ndwi_mask)).unmask(1)
    return img.addBands(zz.rename('NDWI_norm'))


def TF_get(img):
    TFI = img.select('TFI').gt(0.1).And(img.select('TFI').lt(0.5))
    NDWI = img.select('NDWI').lt(0.2)
    TF_binary = TFI.And(NDWI)
    return img.addBands(TF_binary.rename('TF'))

S2_2022 = S2.map(normalized_TFI).map(TF_get)

Norm_S2_2022 = S2_2022.map(Norm)
cloud_sum = S2_2022.select('cloud').toBands().reduce(ee.Reducer.sum())

NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min')
NDWI_mean = S2_2022.select('NDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDWI')
MNDWI_mean = S2_2022.select('MNDWI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_MNDWI')
composit_b8A = S2_2022.select('B8A').reduce(ee.Reducer.sum()).divide(cloud_sum).divide(10000).rename('mean_B8A')
NDVI_mean = S2_2022.select('NDVI').toBands().reduce(ee.Reducer.sum()).divide(cloud_sum).rename('mean_NDVI')
S2_NDWI_min = S2_2022.select('NDWI').reduce(ee.Reducer.percentile([1])).rename('S2_NDWI_min')

S2_NDWI_norm_min = S2_2022.filter(ee.Filter.gt('cloud_per', 60)).select('NDWI_norm').reduce(ee.Reducer.percentile([1])).rename('NDWI_min')
TF_mask = S2_NDWI_norm_min.lt(-0.1).And(Frequency.gt(0.05)).And(Frequency.lt(0.95)).clip(roi).unmask(0).rename('mask')

feature_bands = Frequency.addBands(NDWI_mean).addBands(composit_b8A).addBands(MNDWI_mean).addBands(NDVI_mean).addBands(S2_NDWI_min).addBands(TF_mask)

In [7]:
import concurrent.futures
import os


coords = roi.bounds().coordinates().getInfo()[0]
minLon, minLat = coords[0]
maxLon, maxLat = coords[2]

# 经纬度换算（粗略）
tile_size_deg = 5000 / 111320

tiles = []
lat = minLat
while lat < maxLat:
    lon = minLon
    while lon < maxLon:
        lon2 = min(lon + tile_size_deg, maxLon)
        lat2 = min(lat + tile_size_deg, maxLat)
        tiles.append(ee.Geometry.BBox(lon, lat, lon2, lat2))
        lon += tile_size_deg
    lat += tile_size_deg

print(f"将切成 {len(tiles)} 个切片进行下载...")

output_dir = "G:\GEEdownload"
os.makedirs(output_dir, exist_ok=True)

def export_tile(i_tile):
    i, tile = i_tile
    filename = os.path.join(output_dir, f"tile_{i+1}.tif")
    print(f"开始导出 {filename} ...")
    geemap.ee_export_image(
        feature_bands,
        filename=filename,
        scale=30,
        region=tile,
        file_per_band=False
    )
    print(f"完成导出 {filename}")
    return filename

# 用线程池并发执行导出
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    results = list(executor.map(export_tile, enumerate(tiles)))

print("✅ 所有切片导出完成！")

将切成 104 个切片进行下载...
开始导出 G:\GEEdownload\tile_1.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_2.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_3.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_4.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_5.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_6.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_7.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_8.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_9.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_10.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_11.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_12.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_13.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_14.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_15.tif ...
Generating URL ...
开始导出 G:\GEEdownload\tile_16.tif ...
Generating URL ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ...
Please wait ..

In [8]:
import glob
import rasterio
from rasterio.merge import merge

src_files_to_mosaic = [rasterio.open(fp) for fp in glob.glob(f"{output_dir}/tile_*.tif")]
mosaic, out_trans = merge(src_files_to_mosaic)

out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans
})

merged_path = "G:\GEEdownload\merged_image.tif"
with rasterio.open(merged_path, "w", **out_meta) as dest:
    dest.write(mosaic)

print(f"✅ 已完成拼接，输出文件: {merged_path}")

✅ 已完成拼接，输出文件: G:\GEEdownload\merged_image.tif
